# Prompt Polisher — Full Model Training (Colab T4)

This notebook retrains the **PromptPolisherTransformer** from scratch for **50,000 steps**.

**Architecture:** LLaMA-style decoder (RMSNorm, RoPE, SwiGLU) — 6 layers, 384 hidden, 6 heads, ~35M params

**Pipeline:** Collect corpus → Train tokenizer → Collect SFT data → Train model

**Time:** ~1-1.5 hours total on T4 (training is ~45-55 min)

---
**IMPORTANT:** Set runtime to **GPU → T4** before running.

`Runtime → Change runtime type → T4 GPU`

## Step 0: Setup — Clone repo & install dependencies

In [1]:
# Clone the repository
!git clone https://github.com/Prompt-Polisher-Org/prompt-polisher.git
%cd prompt-polisher

Cloning into 'prompt-polisher'...
remote: Enumerating objects: 807, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 807 (delta 79), reused 133 (delta 57), pack-reused 601 (from 1)
Receiving objects: 100% (807/807), 1.02 MiB | 11.91 MiB/s, done.
Resolving deltas: 100% (313/313), done.
/content/prompt-polisher


In [2]:
# Install exact dependencies
!pip install -q torch>=2.4 sentencepiece>=0.2.0 datasets>=2.19 tqdm>=4.66 tensorboard numpy>=1.26

In [3]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "ERROR: No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
PyTorch: 2.11.0+cu130
CUDA: 13.0


## Step 1: Collect tokenizer corpus (~5 min)

Downloads from HuggingFace:
- OpenAssistant/oasst2
- databricks/databricks-dolly-15k
- sahil2801/CodeAlpaca-20k

Output: `ai/data/corpus/tokenizer_corpus.txt`

In [4]:
!python -m ai.src.tokenizer.collect_corpus

📚 Prompt Polisher — Corpus Collection for Tokenizer Training

📥 Loading: OpenAssistant/oasst2 (split=train)...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'OpenAssistant/oasst2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 100% 10.6k/10.6k [00:00<00:00, 22.0MB/s]

data/train-00000-of-00001-88ba0162028a73(…): downloading bytes:  94% 59.4M/63.5M [00:01<00:00, 68.8MB/s, 4.86MB/s  ]
data/train-00000-of-00001-88ba0162028a73(…): downloading bytes: 100% 62.0M/62.0M [00:01<00:00, 41.3MB/s, 5.75MB/s  ]
data/train-00000-of-00001-88ba0162028a73(…): reconstructing file: 100% 63.5M/63.5M [00:01<00:00, 42.4MB/s, 6.11MB/s  ]

data/validation-00000-of-00001-1deeef95c(…): downloading bytes:   0% 0.00/3.18M [00:00<?, ?B/s]
data/validation-00000-of-00001-1deeef95c(…): downloading bytes: 100% 3.1

In [5]:
# Verify corpus was created
import os
corpus_path = "ai/data/corpus/tokenizer_corpus.txt"
size_mb = os.path.getsize(corpus_path) / (1024 * 1024)
with open(corpus_path, 'r', encoding='utf-8') as f:
    line_count = sum(1 for _ in f)
print(f"Corpus: {size_mb:.1f} MB, {line_count:,} lines")

Corpus: 85.1 MB, 155,323 lines


## Step 2: Train SentencePiece BPE tokenizer (~3 min)

Config (must match exactly):
- Algorithm: BPE
- Vocab size: 32,000
- Special tokens: `<pad>`(0), `<unk>`(1), `<bos>`(2), `<eos>`(3), `<sep>`(4)
- Byte fallback: enabled
- Character coverage: 0.9995

Output: `ai/models/tokenizer/prompt_polisher_bpe.model`

In [6]:
!python -m ai.src.tokenizer.train_tokenizer

🔤 Prompt Polisher — BPE Tokenizer Training
   📁 Corpus:     /content/prompt-polisher/ai/data/corpus/tokenizer_corpus.txt (85.1 MB)
   📁 Output:     /content/prompt-polisher/ai/models/tokenizer/prompt_polisher_bpe.model
   📊 Vocab size: 32,000
   🔧 Model type: BPE
   🧵 Threads:    2

I0000 00:00:1792692252.264032    2693 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /content/prompt-polisher/ai/data/corpus/tokenizer_corpus.txt
  input_format: 
  model_prefix: /content/prompt-polisher/ai/models/tokenizer/prompt_polisher_bpe
  model_type: BPE
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4096
  num_threads: 2
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whi

In [7]:
# Verify tokenizer works
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("ai/models/tokenizer/prompt_polisher_bpe.model")

print(f"Vocab size: {sp.get_piece_size()}")
print(f"BOS ID: {sp.bos_id()}")
print(f"EOS ID: {sp.eos_id()}")
print(f"PAD ID: {sp.pad_id()}")
print(f"UNK ID: {sp.unk_id()}")

# Test encode/decode
test = "Optimize the following prompt to be more effective"
ids = sp.encode(test, out_type=int)
decoded = sp.decode(ids)
print(f"\nTest: '{test}'")
print(f"Token IDs: {ids}")
print(f"Decoded: '{decoded}'")
print(f"Fertility: {len(ids)/len(test.split()):.2f} tokens/word")

Vocab size: 32000
BOS ID: 2
EOS ID: 3
PAD ID: 0
UNK ID: 1

Test: 'Optimize the following prompt to be more effective'
Token IDs: [11933, 864, 278, 1260, 3317, 299, 371, 643, 4523]
Decoded: 'Optimize the following prompt to be more effective'
Fertility: 1.12 tokens/word


## Step 3: Collect SFT training data (~5 min)

**Primary source:** [Junrulu/Prompt_Preference_Dataset](https://huggingface.co/datasets/Junrulu/Prompt_Preference_Dataset) — 30K real prompt rewrites by GPT-4.
Each pair is a genuine rewrite: the raw prompt is completely restructured with role assignments, formatting guidance, and detail requirements.

**Supplementary sources (template-based):**
- databricks/databricks-dolly-15k — general instruction polishing
- sahil2801/CodeAlpaca-20k — code prompt polishing
- Local MATH dataset (optional) — LaTeX math prompt polishing

Target: ~25,000 pairs (deduped, shuffled with seed=42)

Output: `ai/data/sft/splits/train.jsonl`, `val.jsonl`, `test.jsonl` (90/5/5 split)

In [8]:
!python -m ai.src.training.collect_sft_data

[SFT] Prompt Polisher -- SFT Data Collection (v2 - diverse templates)
[WARN] MATH dataset not found at /content/prompt-polisher/FY_dataset/train.csv/train.csv, skipping
[HF] Loading databricks/databricks-dolly-15k...
   [OK] Extracted 14931 pairs from Dolly
[HF] Loading tatsu-lab/alpaca...
README.md: 100% 7.47k/7.47k [00:00<00:00, 17.2MB/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:  99% 24.1M/24.2M [00:00<00:00, 61.1MB/s, 1.10MB/s  ]
data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes: 100% 24.1M/24.1M [00:00<00:00, 42.0MB/s, 2.37MB/s  ]
data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file: 100% 24.2M/24.2M [00:00<00:00, 42.2MB/s, 2.39MB/s  ]
Generating train split: 100% 52002/52002 [00:00<00:00, 279041.81 examples/s]
   [OK] Extracted 52001 pairs from Alpaca
[HF] Loading sahil2801/CodeAlpaca-20k...
   [OK] Extracted 20022 pairs from CodeAlpaca

[STATS] Total collected: 86954 pairs
[STATS] After dedup: 86671 unique pairs
[STATS] Capped at: 

In [9]:
# Verify data splits
import json

for split in ['train', 'val', 'test']:
    path = f"ai/data/sft/splits/{split}.jsonl"
    with open(path, 'r', encoding='utf-8') as f:
        count = sum(1 for _ in f)
    print(f"{split}: {count} examples")

# Show a sample
with open("ai/data/sft/splits/train.jsonl", 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
print(f"\n--- Sample ---")
print(f"Input:  {sample['input'][:100]}...")
print(f"Output: {sample['output'][:100]}...")

train: 10800 examples
val: 600 examples
test: 600 examples

--- Sample ---
Input:  List the five most populous cities in California....
Output: List the five most populous cities in California.

Write clean, production-ready the appropriate lan...


## Step 4: Train the model (~45-55 min on T4)

**Exact config matching `best_model.pt`:**

| Parameter | Value |
|-----------|-------|
| Architecture | 6 layers, 384 hidden, 6 heads, 1024 FFN |
| Parameters | ~35.2M (22.9M unique, weight-tied) |
| Max seq len | 512 |
| Optimizer | AdamW (betas=0.9,0.95, wd=0.01) |
| LR | 3e-4 peak, cosine decay, 500 warmup |
| Batch size | 8 × 4 grad_accum = 32 effective |
| Precision | Mixed FP16 |
| Steps | 50,000 |
| Grad clipping | max_norm 1.0 |

**NOTE:** 50K steps takes ~45-55 min on T4. Keep Colab tab open or save to Google Drive (Step 6) to avoid losing progress.

In [10]:
# Verify model architecture before training
from ai.src.training.config import get_small_config
from ai.src.training.architecture import create_model

config = get_small_config()
model = create_model(config)
param_count = sum(p.numel() for p in model.parameters())
print(config)
print(f"\nExact parameter count: {param_count:,}")
print(f"Expected (weight-tied): 22,909,824")
print(f"In checkpoint state_dict (counts shared weight twice): 35,197,824")
assert param_count == 22909824, f"Parameter mismatch! Got {param_count}"
print("Architecture matches — ready to train!")
del model  # Free memory before training

[MODEL] Created PromptPolisherTransformer
   Layers: 6 | Hidden: 384 | Heads: 6
   Parameters: 22,909,824 (22.9M)
ModelConfig(
  Architecture: 6L / 384D / 6H
  FFN: 1024 (SwiGLU)
  Vocab: 32,000 | Context: 512
  Parameters: ~35.4M
  Training: bs=8×4, lr=0.0003
)

Exact parameter count: 22,909,824
Expected (weight-tied): 22,909,824
In checkpoint state_dict (counts shared weight twice): 35,197,824
Architecture matches — ready to train!


In [11]:
# (Optional) Mount Google Drive BEFORE training so checkpoints save there
# This protects against Colab disconnects during the ~6-8 hour run
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
save_dir = '/content/drive/MyDrive/prompt-polisher-model'
os.makedirs(save_dir, exist_ok=True)

# Symlink checkpoint dir to Drive so saves go there automatically
checkpoint_dir = 'ai/models/checkpoints'
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
os.symlink(save_dir, checkpoint_dir)
print(f"Checkpoints will save directly to Google Drive: {save_dir}")

Mounted at /content/drive
Checkpoints will save directly to Google Drive: /content/drive/MyDrive/prompt-polisher-model


In [12]:
# TRAIN — this takes ~6-8 hours on T4 for 50K steps
# --config small  → 6L/384D/6H (matches best_model.pt architecture exactly)
# --max-steps 50000  → full training run (previous was only 11K = 22%)
!python -m ai.src.training.train --config small --max-steps 50000

Streaming output truncated to the last 5000 lines.
   Step   1750/50000 | Loss: 0.7070 | LR: 2.62e-04 | Tok/s: 24248 | Time: 99s
   Step   1760/50000 | Loss: 0.6640 | LR: 2.64e-04 | Tok/s: 24289 | Time: 99s
   Step   1770/50000 | Loss: 0.5929 | LR: 2.65e-04 | Tok/s: 24307 | Time: 100s
   Step   1780/50000 | Loss: 0.6248 | LR: 2.67e-04 | Tok/s: 24330 | Time: 100s
   Step   1790/50000 | Loss: 0.5814 | LR: 2.68e-04 | Tok/s: 24347 | Time: 101s
   Step   1800/50000 | Loss: 0.5936 | LR: 2.70e-04 | Tok/s: 24363 | Time: 101s
   Step   1810/50000 | Loss: 0.5325 | LR: 2.71e-04 | Tok/s: 24371 | Time: 101s
   Step   1820/50000 | Loss: 0.5010 | LR: 2.73e-04 | Tok/s: 24368 | Time: 102s
   Step   1830/50000 | Loss: 0.7071 | LR: 2.74e-04 | Tok/s: 24405 | Time: 102s
   Step   1840/50000 | Loss: 0.6112 | LR: 2.76e-04 | Tok/s: 24426 | Time: 103s
   Step   1850/50000 | Loss: 0.5240 | LR: 2.77e-04 | Tok/s: 24427 | Time: 103s
   Step   1860/50000 | Loss: 0.5845 | LR: 2.79e-04 | Tok/s: 24438 | Time: 104s
   

In [13]:
# Verify the trained checkpoint
import torch

# The training script saves best_model.pt (best val loss) and step_N.pt checkpoints
import glob
checkpoints = glob.glob("ai/models/checkpoints/*.pt")
print("Saved checkpoints:")
for cp_path in sorted(checkpoints):
    size_mb = os.path.getsize(cp_path) / (1024 * 1024)
    print(f"  {cp_path} ({size_mb:.1f} MB)")

# Load and inspect best checkpoint
best_path = "ai/models/checkpoints/best_model.pt"
if os.path.exists(best_path):
    cp = torch.load(best_path, map_location='cpu', weights_only=False)
    print(f"\n--- best_model.pt ---")
    print(f"Step: {cp['step']}")
    print(f"Loss: {cp['loss']:.4f}")
    print(f"Config layers: {cp['config']['num_layers']}")
    print(f"Config hidden: {cp['config']['hidden_dim']}")
    print(f"Config heads: {cp['config']['num_heads']}")
    sd = cp['model_state_dict']
    total = sum(v.numel() for v in sd.values())
    print(f"Parameters: {total:,}")
else:
    print("WARNING: best_model.pt not found — using latest step checkpoint")

Saved checkpoints:
  ai/models/checkpoints/best_model.pt (262.3 MB)
  ai/models/checkpoints/final_model.pt (262.3 MB)
  ai/models/checkpoints/step_1000.pt (262.3 MB)
  ai/models/checkpoints/step_10000.pt (262.3 MB)
  ai/models/checkpoints/step_11000.pt (262.3 MB)
  ai/models/checkpoints/step_12000.pt (262.3 MB)
  ai/models/checkpoints/step_13000.pt (262.3 MB)
  ai/models/checkpoints/step_14000.pt (262.3 MB)
  ai/models/checkpoints/step_15000.pt (262.3 MB)
  ai/models/checkpoints/step_16000.pt (262.3 MB)
  ai/models/checkpoints/step_17000.pt (262.3 MB)
  ai/models/checkpoints/step_18000.pt (262.3 MB)
  ai/models/checkpoints/step_19000.pt (262.3 MB)
  ai/models/checkpoints/step_2000.pt (262.3 MB)
  ai/models/checkpoints/step_20000.pt (262.3 MB)
  ai/models/checkpoints/step_21000.pt (262.3 MB)
  ai/models/checkpoints/step_22000.pt (262.3 MB)
  ai/models/checkpoints/step_23000.pt (262.3 MB)
  ai/models/checkpoints/step_24000.pt (262.3 MB)
  ai/models/checkpoints/step_25000.pt (262.3 MB)
  

## Step 5: Quick test — generate a sample prompt

In [14]:
from ai.src.inference.engine import InferenceEngine

# Load from the best checkpoint
best_path = "ai/models/checkpoints/best_model.pt"
if not os.path.exists(best_path):
    # Fallback to the latest step checkpoint
    checkpoints = sorted(glob.glob("ai/models/checkpoints/step_*.pt"))
    best_path = checkpoints[-1]

engine = InferenceEngine.from_checkpoint(best_path)

# Test prompts
test_prompts = [
    "Write about dogs",
    "Explain quantum computing",
    "Create a Python sorting function",
    "solve 2 raise to x + 3 cubed = 90",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"INPUT:  {prompt}")
    print(f"-"*60)
    result = engine.generate(prompt, temperature=0.7, max_new_tokens=256)
    print(f"OUTPUT: {result['generated_text']}")
    print(f"Tokens: {result['token_count']} | Latency: {result['latency_ms']:.0f}ms")


INPUT:  Write about dogs
------------------------------------------------------------
OUTPUT: Write about dogs and dogs Length: 300-500 words. Tone: friendly. Structure: clear thesis, specific details over generalizations, memorable closing.<eos> concrete example<eos> concrete example<eos>" concrete example<eos> concrete example<eos> ``` Requirements: - Language: the appropriate language - Include type hints and a docstring - Handle edge cases: invalid types, missing keys - Add 3 test example(s) with expected output<eos> concrete example(s) with expected output<eos> concrete example(s) 4. Key takeaways<eos> concrete example(s) 4. Key takeaways<eos> concrete example(s) 4. Key takeaways<eos> concrete example<eos> concrete example<eos> concrete example<eos> concrete example<eos> concrete example<eos> concrete example<eos><eos><eos><eos><eos><eos> concrete example(s) 4. Key takeaways<eos> layers.<eos><eos><eos><eos><eos><e
Tokens: 256 | Latency: 3566ms

INPUT:  Explain quantum computing
-

## Step 6: Download trained files

You need **BOTH** files on your local machine:

1. `prompt_polisher_bpe.model` → place at `ai/models/tokenizer/`
2. `best_model.pt` → place at `backend/app/models/` (or `ai/models/checkpoints/`)

In [15]:
from google.colab import files

# Download the tokenizer (this is the file you were missing)
print("Downloading tokenizer...")
files.download('ai/models/tokenizer/prompt_polisher_bpe.model')

# Download the trained model
print("Downloading best_model.pt...")
best_path = "ai/models/checkpoints/best_model.pt"
if not os.path.exists(best_path):
    checkpoints = sorted(glob.glob("ai/models/checkpoints/step_*.pt"))
    best_path = checkpoints[-1]
files.download(best_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Optional) Save to Google Drive

So you don't lose the files if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/prompt-polisher-model'
os.makedirs(save_dir, exist_ok=True)

# Copy tokenizer
shutil.copy('ai/models/tokenizer/prompt_polisher_bpe.model', save_dir)
print(f"Tokenizer saved to {save_dir}")

# Copy best model
shutil.copy(best_path, save_dir)
print(f"Model saved to {save_dir}")

# Copy tokenizer config
if os.path.exists('ai/models/tokenizer/tokenizer_config.json'):
    shutil.copy('ai/models/tokenizer/tokenizer_config.json', save_dir)

print(f"\nAll files saved to Google Drive: {save_dir}")
!ls -la {save_dir}

---

## After downloading — Local setup

Place the files:
```
prompt-polisher/
  ai/models/tokenizer/prompt_polisher_bpe.model   ← tokenizer
  backend/app/models/best_model.pt                 ← model
```

Then start the inference server:
```bash
python -m uvicorn ai.src.inference.server:app --host 0.0.0.0 --port 8001
```

It should now say **"Model loaded successfully"** instead of mock mode.